# Memory & Performance Optimization

Companion notebook for the [Memory & Performance lesson](https://ml-viz-ruby.vercel.app/courses/gpu-programming/03-memory-and-performance).

**The idea in one sentence.** Most GPU kernels are **memory-bound**, so performance
is about *moving fewer bytes*: access memory so a warp's 32 loads **coalesce** into
one transaction, **tile** data into fast on-chip memory to reuse it, and use the
**roofline** to know whether you're memory- or compute-limited.

Three tools, from scratch:

- **Coalescing** — contiguous (stride-1) access uses every fetched byte; strided
  access wastes bandwidth.
- **Tiling** — load a block once into shared memory and reuse it, cutting global
  traffic by the tile size.
- **The roofline model** — arithmetic intensity vs the ridge point tells you the
  bottleneck.

We **validate the coalescing and tiling formulas and the roofline crossover**, then
cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})

## 1 — Coalescing efficiency vs. stride

A warp of 32 threads reads addresses `t * stride`. Memory arrives in fixed transactions (a cache
line of `SEG` cells). Efficiency = bytes the threads actually use ÷ bytes the hardware fetched.
Stride 1 is fully coalesced (~100%); larger strides scatter the reads across more transactions.

In [ ]:
def coalescing_efficiency(stride, warp=32, seg=32):
    addresses = np.arange(warp) * stride
    transactions = len(set(addresses // seg))      # distinct cache lines touched
    bytes_fetched = transactions * seg
    return warp / bytes_fetched

strides = [1, 2, 4, 8, 16, 32]
eff = [coalescing_efficiency(s) for s in strides]

fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.bar([str(s) for s in strides], [e * 100 for e in eff], color='#6366f1')
ax.set_xlabel('access stride'); ax.set_ylabel('bandwidth efficiency (%)')
ax.set_title('Coalescing: stride 1 uses all fetched bytes; striding wastes them')
ax.grid(True, alpha=0.3, axis='y')
for b, e in zip(bars, eff):
    ax.text(b.get_x() + b.get_width()/2, e*100 + 1, f'{e*100:.0f}%', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

for s, e in zip(strides, eff):
    print(f"stride {s:2d}: efficiency {e*100:5.1f}%")

### Validate: coalescing efficiency is $1/\text{stride}$ (up to the warp size)

For a warp of 32 threads with access stride $s$, the number of distinct 32-byte
segments touched grows with $s$, so bandwidth efficiency falls as $1/s$ until every
thread hits its own segment (efficiency $1/32$). We check the simulated efficiency
matches that closed form.

In [ ]:
for s in [1, 2, 4, 8, 16, 32]:
    eff = coalescing_efficiency(s)
    expected = 1.0 / s
    print(f'stride {s:2d}: efficiency {eff:.3f}  (1/stride = {expected:.3f})')
    assert abs(eff - expected) < 1e-9, 'coalescing efficiency should be 1/stride'
print('\n✅ stride-1 access is fully coalesced; each doubling of stride halves bandwidth')

## 2 — Tiling cuts global-memory traffic

Naive matmul re-reads each input element from slow global memory once per output it contributes to.
Tiling loads a `T x T` tile into shared memory once and reuses it across the tile, cutting global
reads by ~`T`. We count global-memory reads for an `N x N` matmul both ways.

In [ ]:
def global_reads_naive(N):
    # each of N*N outputs reads N from A and N from B
    return N * N * (2 * N)

def global_reads_tiled(N, T):
    # tiles of size T; each (T x T) output tile loads (N/T) pairs of T x T input tiles once
    tiles = (N // T) ** 2
    loads_per_tile = (N // T) * (2 * T * T)
    return tiles * loads_per_tile

N = 1024
for T in [8, 16, 32]:
    naive = global_reads_naive(N)
    tiled = global_reads_tiled(N, T)
    print(f"N={N}, tile T={T:2d}:  naive={naive/1e9:.2f}G reads  tiled={tiled/1e9:.2f}G  -> {naive/tiled:.1f}x less traffic")

### Validate: tiling cuts global-memory traffic by the tile size

Naive matmul re-reads each input row/column from slow global memory for every
output; tiling loads a $T\times T$ block into fast shared memory once and reuses it,
reducing global reads by a factor of $\approx T$. We confirm the traffic ratio
tracks the tile dimension.

In [ ]:
N = 1024
for T in [8, 16, 32]:
    ratio = global_reads_naive(N) / global_reads_tiled(N, T)
    print(f'tile T={T:2d}: naive/tiled global-read ratio = {ratio:.1f}x  (~T = {T})')
    assert abs(ratio - T) < 1e-6, 'tiling should cut global traffic by ~T'
print('\n✅ a T×T tile reuses each loaded value T times -> ~T× less global-memory traffic')

## 3 — The roofline model

Achievable performance is capped by $\min(P_{\max},\, I \times BW)$ where $I$ is arithmetic intensity
(FLOP/byte). Low-$I$ ops are **memory-bound** (left of the ridge); high-$I$ ops are **compute-bound**
(right). We use representative numbers ($P_{\max}$ = 20 TFLOP/s, $BW$ = 1 TB/s).

In [ ]:
P_max = 20e12     # peak FLOP/s
BW = 1e12         # bytes/s
ridge = P_max / BW
print(f"ridge point I* = P_max / BW = {ridge:.0f} FLOP/byte")

I = np.logspace(-1, 3, 200)
perf = np.minimum(P_max, I * BW)

ops = {
    'vector add (~0.08)': 0.08,
    'activation (~0.5)':  0.5,
    'small matmul (~8)':  8,
    'large matmul (~200)': 200,
}
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.loglog(I, perf / 1e12, color='#818cf8')
ax.axvline(ridge, ls='--', color='#555', label=f'ridge I*={ridge:.0f}')
for name, intensity in ops.items():
    p = min(P_max, intensity * BW)
    bound = 'memory' if intensity < ridge else 'compute'
    c = '#fb7185' if bound == 'memory' else '#2dd4bf'
    ax.plot(intensity, p / 1e12, 'o', color=c, ms=9)
    ax.annotate(f'{name}\n[{bound}-bound]', (intensity, p/1e12),
                textcoords='offset points', xytext=(6, -18), fontsize=7, color=c)
ax.set_xlabel('arithmetic intensity I (FLOP/byte)'); ax.set_ylabel('performance (TFLOP/s)')
ax.set_title('Roofline: memory-bound (left) vs compute-bound (right)')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

### Validate: the roofline crossover is the ridge point

The roofline caps achievable performance at $\min(P_{\max},\, I\cdot BW)$: below the
ridge $I^*=P_{\max}/BW$ you're **memory-bound** (perf $=I\cdot BW$), above it
**compute-bound** (perf $=P_{\max}$). We confirm each operation lands on the right
side of the ridge.

In [ ]:
def roofline(I): return min(P_max, I * BW)
for name, intensity in ops.items():
    perf_pt = roofline(intensity)
    bound = 'memory' if intensity < ridge else 'compute'
    print(f'{name:22s} I={intensity:6.2f} -> {perf_pt/1e12:5.1f} TFLOP/s [{bound}-bound]')
    if intensity < ridge:
        assert np.isclose(perf_pt, intensity * BW), 'below ridge: memory-bound (I*BW)'
    else:
        assert np.isclose(perf_pt, P_max), 'above ridge: compute-bound (P_max)'
print(f'\n✅ ridge point I* = {ridge:.0f} FLOP/byte separates memory- from compute-bound')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **strided / random access** | uncoalesced loads waste most of the bandwidth (efficiency $1/\text{stride}$) |
| **shared-memory bank conflicts** | threads hitting the same bank serialize, negating the tiling win |
| **tile too big** | exceeds shared memory / registers → lower occupancy |
| **optimising the wrong bound** | adding compute to a memory-bound kernel does nothing; raise intensity instead |
| **ignoring the ridge point** | you can't know which lever helps without the roofline analysis |

Demo: fusing elementwise ops raises arithmetic intensity and speeds up a
memory-bound kernel.

In [ ]:
# The practical takeaway: a memory-bound kernel gets FASTER by raising its arithmetic
# intensity (do more FLOPs per byte loaded), not by adding compute. Fusing operations
# is how you do it — one load feeds several ops instead of one.
def speedup_from_fusion(I_before, n_fused, P_max=20e12, BW=1e12):
    ridge = P_max / BW
    I_after = I_before * n_fused                 # reuse the loaded bytes across n_fused ops
    return roofline_after(I_after) / roofline_after(I_before)
def roofline_after(I, P_max=20e12, BW=1e12): return min(P_max, I * BW)
base_I = 0.3                                     # memory-bound elementwise op
for k in [1, 2, 4, 8]:
    su = roofline_after(base_I * k) / roofline_after(base_I)
    print(f'fusing {k} elementwise ops (intensity {base_I}->{base_I*k}): {su:.1f}x faster (while memory-bound)')
print('\nRaising arithmetic intensity moves you UP the memory roofline toward the compute ceiling.')

## ✏️ Your turn

**Exercise.** Implement `roofline_perf(I, P_max, BW)` returning achievable performance, and
`is_memory_bound(I, P_max, BW)` returning `True` when the op is limited by bandwidth (left of the
ridge point $I^* = P_{\max}/BW$).

In [ ]:
def roofline_perf(I, P_max, BW):
    # TODO(you): achievable performance is the lower of the compute and memory ceilings
    return ...

def is_memory_bound(I, P_max, BW):
    # TODO(you): True if intensity is below the ridge point
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert roofline_perf(0.5, 20e12, 1e12) == 0.5e12        # memory-bound: I*BW
assert roofline_perf(200, 20e12, 1e12) == 20e12         # compute-bound: P_max
assert is_memory_bound(0.5, 20e12, 1e12) is True
assert is_memory_bound(200, 20e12, 1e12) is False
assert coalescing_efficiency(1) == 1.0                  # stride 1 fully coalesced
assert coalescing_efficiency(2) == 0.5                  # stride 2 wastes half
print("\u2713 roofline + coalescing checks pass")

<details>
<summary>Solution</summary>

```python
def roofline_perf(I, P_max, BW):
    return min(P_max, I * BW)

def is_memory_bound(I, P_max, BW):
    return I < P_max / BW
```

The ridge point $I^* = P_{\max}/BW$ is where the two ceilings meet. Below it, raising FLOPs does
nothing — you must raise intensity (fuse, tile) or bandwidth (coalesce). Above it, you're limited by
raw arithmetic throughput, where tensor cores and low precision help.

</details>

## Key takeaways

- **Most kernels are memory-bound** — optimisation means moving fewer bytes, not
  adding FLOPs.
- **Coalesce your accesses:** stride-1 uses every fetched byte; efficiency is
  $1/\text{stride}$ (verified). This is the single biggest bandwidth lever.
- **Tile for reuse:** a $T\times T$ shared-memory tile cuts global traffic ~$T\times$
  (verified) — the core of fast matmul.
- **Use the roofline:** below the ridge $I^*=P_{\max}/BW$ you're memory-bound (raise
  intensity via **fusion**); above it, compute-bound (verified crossover).